# 04 — Enterprise & Observability (v0.4)

Demonstra a camada de inventário/observabilidade:

1. `automation/inventory/aws_inventory.py` rodando contra AWS **mockada com
   `moto`** — zero custo, zero credenciais reais, zero chamada de rede à
   AWS de verdade.
2. O schema MySQL (`database/schema.sql`) que registra execuções,
   incidentes e inventário.
3. Como o dashboard (`dashboard/js/app.js`) consome esses dados (via
   `dashboard/js/data.example.json` quando MySQL não está rodando
   localmente).

In [1]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing PROJECT_LOG.md
    is found. Works whether the notebook is executed from notebooks/ (the
    normal case) or from the repo root."""
    p = start.resolve()
    for _ in range(8):
        if (p / "PROJECT_LOG.md").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError("Could not locate project root (PROJECT_LOG.md not found upward from %s)" % start)

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\Caterpillar (Terminar)


## Parte 1 — Inventário AWS mockado (`moto`)

In [2]:
try:
    from moto import mock_aws
    MOTO_AVAILABLE = True
except ImportError:
    MOTO_AVAILABLE = False
    print("moto não está instalado — pulando a demonstração de inventário mockado.")

if MOTO_AVAILABLE:
    import boto3
    from automation.inventory.aws_inventory import get_inventory

    REGION = "sa-east-1"

    @mock_aws
    def build_fake_environment_and_get_inventory():
        # Monta um ambiente AWS fake equivalente ao que o Terraform (Notebook 01)
        # criaria de verdade: 1 VPC, 1 subnet, 2 EC2 (web-01, web-02) e 1 RDS MySQL.
        ec2 = boto3.client("ec2", region_name=REGION)
        vpc = ec2.create_vpc(CidrBlock="10.0.0.0/16")["Vpc"]
        subnet = ec2.create_subnet(
            VpcId=vpc["VpcId"], CidrBlock="10.0.0.0/24", AvailabilityZone=f"{REGION}a"
        )["Subnet"]

        for name in ("linux-web-01", "linux-web-02"):
            ec2.run_instances(
                ImageId="ami-0123456789abcdef0",
                MinCount=1, MaxCount=1,
                InstanceType="t3.micro",
                SubnetId=subnet["SubnetId"],
                TagSpecifications=[{"ResourceType": "instance", "Tags": [{"Key": "Name", "Value": name}]}],
            )

        rds = boto3.client("rds", region_name=REGION)
        rds.create_db_instance(
            DBInstanceIdentifier="enterprise-automation-dev",
            DBInstanceClass="db.t3.micro",
            Engine="mysql",
            MasterUsername="admin",
            MasterUserPassword="ChangeMe123!",
            AllocatedStorage=20,
            Tags=[{"Key": "Name", "Value": "enterprise-automation-dev"}],
        )

        return get_inventory(region=REGION)

    inventory = build_fake_environment_and_get_inventory()
    print(f"region: {inventory['region']}")
    print(f"generated_at: {inventory['generated_at']}")
    print(f"\nEC2 ({len(inventory['ec2'])}):")
    for i in inventory["ec2"]:
        print(f"  {i['id']:20s} {i['name']:20s} {i['instance_type']:10s} {i['state']}")
    print(f"\nRDS ({len(inventory['rds'])}):")
    for r in inventory["rds"]:
        print(f"  {r['id']:28s} {r['engine']:8s} {r['instance_type']:14s} {r['state']}")

region: sa-east-1
generated_at: 2026-08-19T23:23:10.363067+00:00

EC2 (2):
  i-d249539932e874522  linux-web-01         t3.micro   running
  i-c68e9bc9669340d5a  linux-web-02         t3.micro   running

RDS (1):
  enterprise-automation-dev    mysql    db.t3.micro    available


Importante: nenhuma credencial AWS real foi usada e nenhuma chamada saiu
para a AWS de verdade — `@mock_aws` intercepta as chamadas boto3 dentro do
processo. `aws_inventory.get_inventory()` roda **exatamente o mesmo código**
que rodaria contra o ambiente `dev` real, uma vez que o Terraform (Notebook
01) tiver sido aplicado — basta trocar as credenciais.

## Parte 2 — Schema MySQL (`database/schema.sql`)

In [3]:
import re

schema_path = PROJECT_ROOT / "database" / "schema.sql"
schema_sql = schema_path.read_text(encoding="utf-8")

table_names = re.findall(r"CREATE TABLE IF NOT EXISTS `(\w+)`", schema_sql)
print(f"Tabelas declaradas em {schema_path.name}: {table_names}")

Tabelas declaradas em schema.sql: ['executions', 'incidents', 'inventory']


In [4]:
def print_table_columns(sql_text: str, table_name: str):
    start = sql_text.find(f"CREATE TABLE IF NOT EXISTS `{table_name}`")
    end = sql_text.find(") ENGINE=", start)
    block = sql_text[start:end]
    col_lines = [l.strip() for l in block.splitlines() if l.strip().startswith("`")]
    print(f"\n{table_name}:")
    for line in col_lines:
        print(f"  {line}")

for t in table_names:
    print_table_columns(schema_sql, t)


executions:
  `id`             BIGINT UNSIGNED  NOT NULL AUTO_INCREMENT,
  `tool`           ENUM('terraform', 'ansible', 'powershell', 'python')
  `action`         VARCHAR(255)     NOT NULL,
  `environment`    VARCHAR(50)      NOT NULL,
  `status`         ENUM('success', 'failed') NOT NULL,
  `started_at`     DATETIME(3)      NOT NULL,
  `finished_at`    DATETIME(3)      NULL,
  `output_summary` TEXT             NULL,
  `created_at`     TIMESTAMP        NOT NULL DEFAULT CURRENT_TIMESTAMP,

incidents:
  `id`                     BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  `host`                   VARCHAR(255)    NOT NULL,
  `environment`            VARCHAR(50)     NOT NULL,
  `problem`                VARCHAR(500)    NOT NULL,
  `root_cause`             TEXT            NULL,
  `action`                 VARCHAR(500)    NULL,
  `validation`             VARCHAR(500)    NULL,
  `recovery_time_seconds`  INT UNSIGNED    NULL,
  `status`                 ENUM('OPEN', 'RESOLVED') NOT NULL DEFAULT '

Contrato de dados relevante para o self-healing (Notebook 05): a tabela
`incidents` usa exatamente as chaves do exemplo do escopo (`host`,
`environment`, `problem`, `root_cause`, `action`, `validation`,
`recovery_time_seconds`, `status`) — combinado entre a trilha de automação
Python e a trilha de banco de dados.

## Parte 3 — MySQL local (opcional) ou dados de exemplo

In [5]:
"""
Tenta conectar a um MySQL local (docker compose em database/docker-compose.yml).
Se não estiver rodando, cai graciosamente para os dados de exemplo, que é
exatamente o mesmo comportamento do dashboard (dashboard/js/app.js lê
data.example.json quando não há API/DB configurada).
"""
mysql_available = False
try:
    import pymysql  # type: ignore
    conn = pymysql.connect(host="127.0.0.1", port=3306, user="root", password="", connect_timeout=2)
    conn.close()
    mysql_available = True
except Exception as exc:
    print(f"MySQL não está rodando localmente — usando dados de exemplo. ({exc.__class__.__name__}: {exc})")

if mysql_available:
    print("MySQL local detectado (não usado neste notebook para evitar dependência de estado externo).")

MySQL não está rodando localmente — usando dados de exemplo. (ModuleNotFoundError: No module named 'pymysql')


## Parte 4 — Como o dashboard consome os dados (`dashboard/js/data.example.json`)

In [6]:
import json

data_path = PROJECT_ROOT / "dashboard" / "js" / "data.example.json"
if data_path.exists():
    dashboard_data = json.loads(data_path.read_text(encoding="utf-8"))
    print(f"generated_at: {dashboard_data.get('generated_at')}")
    for key in ("executions", "incidents", "inventory"):
        if key in dashboard_data:
            print(f"{key}: {len(dashboard_data[key])} registro(s) de exemplo")
else:
    print("dashboard/js/data.example.json ainda não existe neste checkout.")

generated_at: 2026-08-19T08:00:00Z
executions: 5 registro(s) de exemplo
incidents: 5 registro(s) de exemplo
inventory: 5 registro(s) de exemplo


In [7]:
app_js_path = PROJECT_ROOT / "dashboard" / "js" / "app.js"
if app_js_path.exists():
    js_src = app_js_path.read_text(encoding="utf-8")
    fetch_lines = [l for l in js_src.splitlines() if "fetch(" in l or "data.example.json" in l or "async function" in l]
    print("Trechos relevantes de app.js (como os dados chegam ao dashboard):")
    for l in fetch_lines[:15]:
        print(f"  {l.strip()}")
else:
    print("dashboard/js/app.js ainda não existe neste checkout.")

Trechos relevantes de app.js (como os dados chegam ao dashboard):
  * Data source: js/data.example.json (fetched via $.getJSON), shaped after
  const DATA_URL = "js/data.example.json";


## Resumo

- Inventário AWS (EC2 + RDS) foi gerado com `automation/inventory/aws_inventory.py`
  contra um ambiente 100% mockado via `moto` — o mesmo código funciona
  contra a AWS real sem nenhuma alteração.
- O schema MySQL foi lido e as 3 tabelas (`executions`, `incidents`,
  `inventory`) descritas coluna a coluna.
- O dashboard (estático, HTML/JS/Bootstrap) lê `data.example.json` como
  modo demo — o mesmo formato que uma API real (ou consulta direta ao
  MySQL) alimentaria em produção.